# Logistic Regression Timescale Classifier — Cross-Family Comparison

Train and compare four Logistic Regression (LR) feature sets for predicting the evolutionary timescale of a sequence triplet across three protein families: Chorismate Mutase, Beta-Lactamase, and PF00072.

**Feature sets evaluated:**
| # | Features | Dimensionality |
|---|----------|----------------|
| 1 | `matched_positions` (Hamming mismatch vector) | L |
| 2 | `matched_positions` + `CDE_0_mean` (scalar) | L + 1 |
| 3 | `matched_positions` + `CDE_T_mean` (scalar) | L + 1 |
| 4 | `matched_positions` + `CDE_0_mean` + `CDE_T_mean` | L + 2 |
| 5 | `matched_positions` + full position-wise `CDE_0` + `CDE_T` | 3L |

Results are visualised as grouped bar charts stratified by protein family.

## Setup

Import libraries, configure paths, hyperparameters, and protein-family metadata.

In [ ]:
# ==============================================================================
# IMPORTS AND CONFIGURATION
# ==============================================================================
import gc
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

from adabmDCA.dataset import DatasetDCA
from adabmDCA.utils import get_device, get_dtype

# Make paths work both when the notebook is run from repo root and from paper_notebooks.
if not Path("generated_data").exists() and Path("paper_notebooks/generated_data").exists():
    os.chdir("paper_notebooks")

device = get_device("cpu")
dtype = get_dtype("float32")

data_type = "ACDEFGHIKLMNPQRSTVWY-"
timescales = ["10e1", "10e2", "10e3", "10e4", "10e5", "10e6"]
n_seqs = 30_000
train_ratio = 0.8
random_seed = 42

output_path = Path("immagini_paper/logistic_regression_full")
output_path.mkdir(parents=True, exist_ok=True)

families = {
    "Chorismate Mutase": {
        "data_path": Path("generated_data/CM/train_validation"),
        "cde_path": Path("generated_data/CM/train_validation/full_cde_train"),
    },
    "Betalactamase": {
        "data_path": Path("generated_data/betalactamase/train_validation"),
        "cde_path": Path("generated_data/betalactamase/train_validation/full_cde_train"),
    },
    "PF00072_2nd": {
        "data_path": Path("generated_data/PF00072_2nd/train_validation"),
        "cde_path": Path("generated_data/PF00072_2nd/train_validation/full_cde_train"),
    },
}

base_feature_sets = [
    (["matched_sum"], "matched_sum"),
    (["matched_sum", "CDE_0_mean"], "matched_sum_CDE0_mean"),
    (["matched_sum", "CDE_T_mean"], "matched_sum_CDET_mean"),
    (["matched_sum", "CDE_0_mean", "CDE_T_mean"], "matched_sum_CDE0_mean_CDET_mean"),
]

full_feature_set = ("matched_positions_full_cde", "matched_positions_CDE0_CDET_full")
feature_sets = base_feature_sets + [full_feature_set]
base_method_names = [method_name for _, method_name in base_feature_sets]
extended_method_names = [method_name for _, method_name in feature_sets]



## Data Loading and Feature Building

For each protein family and timescale, load the test FASTA sequences and compute:
- **Position-wise mismatch vector** between the initial and final segment.
- **CDE scalar features** (mean CDE of initial and final segment) loaded from precomputed `.npy` files.

All features and labels are assembled into a single Pandas DataFrame.

In [ ]:
# ==============================================================================
# DATA LOADING AND FEATURE BUILDING
# ==============================================================================
def parse_timescale(ts):
    m = re.match(r"(\d+)e(\d+)", ts)
    if m is None:
        raise ValueError(f"Cannot parse timescale: {ts}")
    base, exp = int(m.group(1)), int(m.group(2))
    return base ** exp


def compute_distance_vector(encoded_triplets):
    """Return the position-wise binary mismatch vector between time 0 and time T."""
    l = encoded_triplets.shape[1] // 3
    time_0, time_t, _ = encoded_triplets.split(l, dim=1)
    return (time_0 != time_t).cpu().numpy().astype(np.float32)


def load_cde_data(cde_path):
    required = ["CDE_0.npy", "CDE_0_mean.npy", "CDE_T.npy", "CDE_T_mean.npy"]
    missing = [name for name in required if not (cde_path / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing CDE files in {cde_path}: {missing}")

    return {
        "CDE_0": np.load(cde_path / "CDE_0.npy", allow_pickle=True).item(),
        "CDE_0_mean": np.load(cde_path / "CDE_0_mean.npy", allow_pickle=True).item(),
        "CDE_T": np.load(cde_path / "CDE_T.npy", allow_pickle=True).item(),
        "CDE_T_mean": np.load(cde_path / "CDE_T_mean.npy", allow_pickle=True).item(),
    }


def build_feature_dataframe(family_name, ts, encoded, cde_data, n_available):
    l = encoded.shape[1] // 3
    dist_cols = [f"dist_{j}" for j in range(l)]
    cde_0_cols = [f"CDE_0_{j}" for j in range(l)]
    cde_t_cols = [f"CDE_T_{j}" for j in range(l)]

    distance_vector = compute_distance_vector(encoded[:n_available])
    cde_0 = np.asarray(cde_data["CDE_0"][ts], dtype=np.float32)[:n_available].reshape(n_available, l)
    cde_t = np.asarray(cde_data["CDE_T"][ts], dtype=np.float32)[:n_available].reshape(n_available, l)
    cde_0_mean = np.asarray(cde_data["CDE_0_mean"][ts], dtype=np.float32)[:n_available]
    cde_t_mean = np.asarray(cde_data["CDE_T_mean"][ts], dtype=np.float32)[:n_available]

    feature_matrix = np.concatenate(
        [
            distance_vector,
            cde_0,
            cde_t,
            distance_vector.sum(axis=1, keepdims=True),
            cde_0_mean[:, None],
            cde_t_mean[:, None],
        ],
        axis=1,
    )
    feature_columns = dist_cols + cde_0_cols + cde_t_cols + ["matched_sum", "CDE_0_mean", "CDE_T_mean"]

    df = pd.DataFrame(feature_matrix, columns=feature_columns)
    df["family"] = family_name
    df["timescale"] = ts
    return df


def load_family_dataframe(family_name, config):
    data_path = config["data_path"]
    cde_path = config["cde_path"]

    if not data_path.exists():
        raise FileNotFoundError(f"Data directory not found for {family_name}: {data_path}")
    if not cde_path.exists():
        raise FileNotFoundError(f"CDE directory not found for {family_name}: {cde_path}")

    cde_data = load_cde_data(cde_path)
    rows = []

    for ts in timescales:
        fasta_path = data_path / f"{ts}_train.fasta"
        if not fasta_path.exists():
            raise FileNotFoundError(f"Missing FASTA for {family_name}, {ts}: {fasta_path}")

        ds = DatasetDCA(
            path_data=str(fasta_path),
            alphabet=data_type,
            device=device,
            dtype=dtype,
            no_reweighting=True,
        )

        n_available = min(
            ds.data.shape[0],
            len(cde_data["CDE_0"][ts]),
            len(cde_data["CDE_T"][ts]),
            len(cde_data["CDE_0_mean"][ts]),
            len(cde_data["CDE_T_mean"][ts]),
            n_seqs,
        )

        rows.append(build_feature_dataframe(family_name, ts, ds.data, cde_data, n_available))

        print(f"{family_name} | {ts}: loaded {n_available} samples")
        del ds
        gc.collect()

    df = pd.concat(rows, ignore_index=True)
    ordered_timescales = sorted(df["timescale"].unique(), key=parse_timescale)
    timescale_to_index = {ts: i for i, ts in enumerate(ordered_timescales)}
    df["class_id"] = df["timescale"].map(timescale_to_index).astype(int)
    return df, ordered_timescales


## Train/Test Split

Split each timescale dataset into train and test subsets using a fixed random seed, maintaining consistent class proportions across the split.

In [ ]:
# ==============================================================================
# TRAIN / TEST SPLIT
# ==============================================================================
def split_family_dataframe(df, seed=random_seed):
    train_parts = []
    test_parts = []

    for ts_idx, ts in enumerate(timescales):
        ts_df = df[df["timescale"] == ts]
        rng = np.random.default_rng(seed + ts_idx)
        indices = rng.permutation(ts_df.index.to_numpy())
        split_idx = int(len(indices) * train_ratio)
        train_parts.append(df.loc[indices[:split_idx]])
        test_parts.append(df.loc[indices[split_idx:]])

    return pd.concat(train_parts), pd.concat(test_parts)


family_splits = {}

for family_name, config in families.items():
    print("\n" + "#" * 80)
    print(f"LOADING AND SPLITTING FAMILY: {family_name}")
    print("#" * 80)

    family_df, ordered_timescales = load_family_dataframe(family_name, config)
    train_df, test_df = split_family_dataframe(family_df)

    family_splits[family_name] = {
        "train": train_df,
        "test": test_df,
        "ordered_timescales": ordered_timescales,
    }

    print(f"{family_name}: train={len(train_df)}, test={len(test_df)}")
    del family_df
    gc.collect()


## Logistic Regression Training

Train one LR classifier per feature set (using L2 regularisation and liblinear solver). Evaluate test accuracy and collect per-family, per-timescale classification reports.

In [ ]:
# ==============================================================================
# LOGISTIC REGRESSION TRAINING
# ==============================================================================
def resolve_feature_columns(df, feature_spec):
    if feature_spec == "matched_positions_full_cde":
        dist_cols = sorted(
            [col for col in df.columns if re.fullmatch(r"dist_\d+", col)],
            key=lambda col: int(col.split("_")[-1]),
        )
        cde_0_cols = sorted(
            [col for col in df.columns if re.fullmatch(r"CDE_0_\d+", col)],
            key=lambda col: int(col.split("_")[-1]),
        )
        cde_t_cols = sorted(
            [col for col in df.columns if re.fullmatch(r"CDE_T_\d+", col)],
            key=lambda col: int(col.split("_")[-1]),
        )
        return dist_cols + cde_0_cols + cde_t_cols

    return feature_spec


def train_family_models(family_name, train_df, test_df, ordered_timescales):
    y_train = train_df["class_id"].to_numpy()
    y_test = test_df["class_id"].to_numpy()

    family_results = []
    models = {}

    for feature_spec, method_name in feature_sets:
        features = resolve_feature_columns(train_df, feature_spec)

        print("\n" + "=" * 70)
        print(f"{family_name} | Logistic Regression | {method_name}")
        print(f"Number of features: {len(features)}")
        print("=" * 70)

        X_train = train_df[features].to_numpy(dtype=np.float32)
        X_test = test_df[features].to_numpy(dtype=np.float32)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = LogisticRegression(max_iter=2000, random_state=random_seed, solver="lbfgs")
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)
        accuracy = accuracy_score(y_test, y_pred)

        print(f"Test accuracy: {accuracy:.4f}")
        print(classification_report(y_test, y_pred, target_names=ordered_timescales))

        family_results.append({
            "Family": family_name,
            "Method": method_name,
            "Num Features": len(features),
            "Accuracy": accuracy,
        })
        models[method_name] = {"model": model, "scaler": scaler, "features": features}

        del X_train, X_test, X_train_scaled, X_test_scaled
        gc.collect()

    return family_results, models


all_results = []
all_models = {}

for family_name, split_data in family_splits.items():
    family_results, family_models = train_family_models(
        family_name=family_name,
        train_df=split_data["train"],
        test_df=split_data["test"],
        ordered_timescales=split_data["ordered_timescales"],
    )

    all_results.extend(family_results)
    all_models[family_name] = family_models
    gc.collect()

results_df = pd.DataFrame(all_results)
results_df


## Results: Accuracy Bar Plot

Plot grouped bar charts comparing test accuracy across feature sets and protein families.

In [ ]:
# ==============================================================================
# FINAL BARPLOT
# ==============================================================================
# Plot parameters: edit these values manually to tune the figure.
figure_width = 12
figure_height = 7
figure_dpi = 300

ylim_min = 0.6
ylim_max = 0.85

bar_color = "#4C78A8"
bar_alpha = 0.9
method_alphas = [0.35, 0.55, 0.75, 1.0]
bar_edge_color = "white"
bar_edge_linewidth = 0.8
bar_label_format = "%.3f"
bar_label_padding = 3
bar_label_fontsize = 14

x_label = "Protein Family"
y_label = "Accuracy"
plot_title = ""
legend_title = "Feature Set"
legend_location = "best"
legend_frameon = False

x_label_fontsize = 26
y_label_fontsize = 26
title_fontsize = 16
title_pad = 15
tick_label_fontsize = 20
legend_fontsize = 18
legend_title_fontsize = 20

grid_alpha = 0.3
grid_linestyle = "--"

tight_layout = True
save_bbox_inches = "tight"

method_labels = {
    "matched_sum": r"$d$",
    "matched_sum_CDE0_mean": r"$d + \overline{\mathrm{CDE}}_{\mathrm{start}}$",
    "matched_sum_CDET_mean": r"$d + \overline{\mathrm{CDE}}_{\mathrm{end}}$",
    "matched_sum_CDE0_mean_CDET_mean": r"$d + \overline{\mathrm{CDE}}_{\mathrm{start}} + \overline{\mathrm{CDE}}_{\mathrm{end}}$",
}

plot_df = results_df[results_df["Method"].isin(base_method_names)].copy()
plot_df["Method Label"] = plot_df["Method"].map(method_labels)
plot_df["Family"] = plot_df["Family"].replace({"PF00072_2nd": "RR domain"})
method_palette = {label: bar_color for label in method_labels.values()}
method_alpha_map = dict(zip(method_labels.values(), method_alphas))

csv_path = output_path / "logistic_regression_full_results.csv"
plot_df.to_csv(csv_path, index=False)

plt.figure(figsize=(figure_width, figure_height))
ax = sns.barplot(
    data=plot_df,
    x="Family",
    y="Accuracy",
    hue="Method Label",
    palette=method_palette,
    edgecolor=bar_edge_color,
    linewidth=bar_edge_linewidth,
    alpha=bar_alpha,
)

ax.set_xlabel(x_label, fontsize=x_label_fontsize)
ax.set_ylabel(y_label, fontsize=y_label_fontsize)
ax.set_title(plot_title, fontsize=title_fontsize, pad=title_pad)
ax.set_ylim(ylim_min, ylim_max)
ax.tick_params(axis="both", labelsize=tick_label_fontsize)
ax.grid(axis="y", alpha=grid_alpha, linestyle=grid_linestyle)
for container, method_label in zip(ax.containers, method_labels.values()):
    for patch in container.patches:
        patch.set_alpha(method_alpha_map[method_label])

handles, labels = ax.get_legend_handles_labels()
for handle, label in zip(handles, labels):
    handle.set_facecolor(bar_color)
    handle.set_alpha(method_alpha_map[label])
    handle.set_edgecolor(bar_edge_color)
    handle.set_linewidth(bar_edge_linewidth)

ax.legend(
    handles=handles,
    labels=labels,
    title=legend_title,
    fontsize=legend_fontsize,
    title_fontsize=legend_title_fontsize,
    loc=legend_location,
    frameon=legend_frameon,
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt=bar_label_format,
        padding=bar_label_padding,
        fontsize=bar_label_fontsize,
    )

if tight_layout:
    plt.tight_layout()

figure_path = output_path / "logistic_regression_full_barplot.pdf"
plt.savefig(figure_path, dpi=figure_dpi, bbox_inches=save_bbox_inches)
plt.show()

print(f"Saved results to: {csv_path}")
print(f"Saved barplot to: {figure_path}")










## Extended Results: Full Position-wise CDE

Extend the bar plot to include the full position-wise CDE feature set (3L features), showing whether fine-grained CDE information improves classification beyond the scalar summary.

In [ ]:
# ==============================================================================
# EXTENDED BARPLOT WITH FULL POSITION-WISE CDE METHOD
# ==============================================================================
# Plot parameters: edit these values manually to tune the extended figure.
ext_figure_width = 13
ext_figure_height = 7
ext_figure_dpi = 300

ext_ylim_min = ylim_min
ext_ylim_max = ylim_max

ext_bar_color = bar_color
ext_bar_alpha = bar_alpha
ext_method_alphas = [0.25, 0.42, 0.59, 0.76, 1.0]
ext_bar_edge_color = bar_edge_color
ext_bar_edge_linewidth = bar_edge_linewidth
ext_bar_label_format = bar_label_format
ext_bar_label_padding = bar_label_padding
ext_bar_label_fontsize = bar_label_fontsize

ext_x_label = x_label
ext_y_label = y_label
ext_plot_title = plot_title
ext_legend_title = legend_title
ext_legend_location = legend_location
ext_legend_frameon = legend_frameon

ext_x_label_fontsize = x_label_fontsize
ext_y_label_fontsize = y_label_fontsize
ext_title_fontsize = title_fontsize
ext_title_pad = title_pad
ext_tick_label_fontsize = tick_label_fontsize
ext_legend_fontsize = legend_fontsize
ext_legend_title_fontsize = legend_title_fontsize

ext_grid_alpha = grid_alpha
ext_grid_linestyle = grid_linestyle

extended_method_labels = {
    **method_labels,
    "matched_positions_CDE0_CDET_full": r"$\mathbf{v} + \mathrm{CDE}_{\mathrm{start}} + \mathrm{CDE}_{\mathrm{end}}$",
}

extended_plot_df = results_df[results_df["Method"].isin(extended_method_names)].copy()
extended_plot_df["Method Label"] = extended_plot_df["Method"].map(extended_method_labels)
extended_plot_df["Family"] = extended_plot_df["Family"].replace({"PF00072_2nd": "RR domain"})
extended_method_palette = {label: ext_bar_color for label in extended_method_labels.values()}
extended_method_alpha_map = dict(zip(extended_method_labels.values(), ext_method_alphas))

extended_csv_path = output_path / "logistic_regression_full_results_with_full_cde.csv"
extended_plot_df.to_csv(extended_csv_path, index=False)

plt.figure(figsize=(ext_figure_width, ext_figure_height))
ext_ax = sns.barplot(
    data=extended_plot_df,
    x="Family",
    y="Accuracy",
    hue="Method Label",
    palette=extended_method_palette,
    edgecolor=ext_bar_edge_color,
    linewidth=ext_bar_edge_linewidth,
    alpha=ext_bar_alpha,
)

ext_ax.set_xlabel(ext_x_label, fontsize=ext_x_label_fontsize)
ext_ax.set_ylabel(ext_y_label, fontsize=ext_y_label_fontsize)
ext_ax.set_title(ext_plot_title, fontsize=ext_title_fontsize, pad=ext_title_pad)
ext_ax.set_ylim(ext_ylim_min, ext_ylim_max)
ext_ax.tick_params(axis="both", labelsize=ext_tick_label_fontsize)
ext_ax.grid(axis="y", alpha=ext_grid_alpha, linestyle=ext_grid_linestyle)

for container, method_label in zip(ext_ax.containers, extended_method_labels.values()):
    for patch in container.patches:
        patch.set_alpha(extended_method_alpha_map[method_label])

handles, labels = ext_ax.get_legend_handles_labels()
for handle, label in zip(handles, labels):
    handle.set_facecolor(ext_bar_color)
    handle.set_alpha(extended_method_alpha_map[label])
    handle.set_edgecolor(ext_bar_edge_color)
    handle.set_linewidth(ext_bar_edge_linewidth)

ext_ax.legend(
    handles=handles,
    labels=labels,
    title=ext_legend_title,
    fontsize=ext_legend_fontsize,
    title_fontsize=ext_legend_title_fontsize,
    loc=ext_legend_location,
    frameon=ext_legend_frameon,
)

for container in ext_ax.containers:
    ext_ax.bar_label(
        container,
        fmt=ext_bar_label_format,
        padding=ext_bar_label_padding,
        fontsize=ext_bar_label_fontsize,
    )

if tight_layout:
    plt.tight_layout()

extended_figure_path = output_path / "logistic_regression_full_barplot_with_full_cde.pdf"
plt.savefig(extended_figure_path, dpi=ext_figure_dpi, bbox_inches=save_bbox_inches)
plt.show()

print(f"Saved extended results to: {extended_csv_path}")
print(f"Saved extended barplot to: {extended_figure_path}")

